# Bangalore Real Estate Price Prediction

This notebook cleans Bangalore property-listing data, builds regression models to estimate listed property prices, and evaluates predictions on a held-out test set. The target variable `price` is expressed in the units provided by the source dataset (commonly lakhs); it is not converted to another currency or unit.

## 1. Setup and load data

The source dataset `banglore.csv` is included beside this notebook. All output artifacts are saved under `output/`.

In [ ]:
from pathlib import Path
import re
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
import joblib

DATA_PATH = Path("banglore.csv")
OUTPUT_DIR = Path("output")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Place the dataset beside this notebook.")

raw_df = pd.read_csv(DATA_PATH)
print(f"Loaded {raw_df.shape[0]:,} rows and {raw_df.shape[1]:,} columns.")
display(raw_df.head())

## 2. Inspect and prepare fields

Keep the fields used by the model: location, size, total square footage, bathrooms, and price. Convert text values to numeric fields and report rows that cannot be parsed.

In [ ]:
required = ["location", "size", "total_sqft", "bath", "price"]
missing_columns = sorted(set(required) - set(raw_df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing from source data: {missing_columns}")

df = raw_df[required].copy()
df["location"] = df["location"].astype("string").str.strip()
df["size"] = df["size"].astype("string").str.strip()

def parse_size(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(\d+(?:\.\d+)?)", str(value))
    return float(match.group(1)) if match else np.nan

def parse_sqft(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().replace(",", "")
    # The dataset contains ranges such as '2100 - 2850'; use their midpoint.
    range_match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)", text)
    if range_match:
        return (float(range_match.group(1)) + float(range_match.group(2))) / 2
    # Values with non-square-foot units are left missing rather than mislabelled as sqft.
    if not re.fullmatch(r"\d+(?:\.\d+)?", text):
        return np.nan
    return float(text)

df["size"] = df["size"].map(parse_size)
df["total_sqft"] = df["total_sqft"].map(parse_sqft)
for column in ["bath", "price"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Missing values after parsing:")
display(df.isna().sum().to_frame("missing_count"))
rows_before = len(df)
df = df.dropna(subset=required).copy()
df = df[
    df["location"].ne("")
    & df["size"].gt(0)
    & df["total_sqft"].gt(0)
    & df["bath"].gt(0)
    & df["price"].gt(0)
].copy()
print(f"Rows retained: {len(df):,} of {rows_before:,} after required-field and positive-value checks.")
display(df.head())

## 3. Exploratory checks

Review the cleaned distributions before modelling. These are descriptive summaries; the notebook does not use full-dataset target-based outlier filtering before the train/test split.

In [ ]:
display(df[["size", "total_sqft", "bath", "price"]].describe().T)

location_counts = df["location"].value_counts().head(15).sort_values()
ax = location_counts.plot(kind="barh", figsize=(9, 6))
ax.set_title("Top 15 Locations by Listing Count")
ax.set_xlabel("Listings")
ax.set_ylabel("Location")
plt.tight_layout()
plt.show()

ax = df.plot.scatter(x="total_sqft", y="price", alpha=0.35, figsize=(8, 5))
ax.set_title("Listed Price vs Total Square Footage")
plt.tight_layout()
plt.show()

## 4. Train/test split and preprocessing

The split occurs before model fitting. Numeric imputation/scaling and categorical imputation/one-hot encoding are placed inside a scikit-learn pipeline, so preprocessing is fitted within each training fold. One-hot encoding ignores locations not seen during training.

In [ ]:
feature_columns = ["location", "size", "total_sqft", "bath"]
target_column = "price"
X = df[feature_columns].copy()
y = df[target_column].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=20
)

numeric_features = ["size", "total_sqft", "bath"]
categorical_features = ["location"]

def make_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ])

print(f"Training rows: {len(X_train):,}")
print(f"Test rows held out for final evaluation: {len(X_test):,}")

## 5. Compare regression models

Candidate models are selected using mean five-fold cross-validation R² on the training set. The held-out test set is used only after model selection and is reported with R², mean absolute error (MAE), and root mean squared error (RMSE). R² is a goodness-of-fit metric, not classification accuracy.

In [ ]:
model_definitions = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(alpha=0.01, max_iter=20_000),
    "Decision Tree": DecisionTreeRegressor(
        max_depth=20, min_samples_leaf=5, random_state=20
    ),
}

candidate_pipelines = {}
cv_rows = []
for name, estimator in model_definitions.items():
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", estimator),
    ])
    cv_scores = cross_val_score(
        pipeline, X_train, y_train, cv=5, scoring="r2", n_jobs=1
    )
    candidate_pipelines[name] = pipeline
    cv_rows.append({
        "Model": name,
        "Mean CV R2": float(cv_scores.mean()),
        "CV R2 Std": float(cv_scores.std()),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("Mean CV R2", ascending=False)
display(cv_results)

best_model_name = cv_results.iloc[0]["Model"]
best_model = candidate_pipelines[best_model_name]
best_model.fit(X_train, y_train)

# The held-out test set is used only once, after model selection.
test_predictions = best_model.predict(X_test)
test_metrics = pd.DataFrame([{
    "Selected Model": best_model_name,
    "Test R2": r2_score(y_test, test_predictions),
    "Test MAE": mean_absolute_error(y_test, test_predictions),
    "Test RMSE": float(np.sqrt(mean_squared_error(y_test, test_predictions))),
}])
display(test_metrics)
print(f"Selected model by mean training-fold CV R2: {best_model_name}")

## 6. Inspect test-set predictions

The sample below compares observed and predicted values for held-out test rows. MAE and RMSE are in the same units as the source dataset's price column.

In [ ]:
predictions = best_model.predict(X_test)
prediction_sample = X_test.copy()
prediction_sample["Actual Price"] = y_test
prediction_sample["Predicted Price"] = predictions
prediction_sample["Absolute Error"] = (prediction_sample["Actual Price"] - prediction_sample["Predicted Price"]).abs()
display(prediction_sample.head(15))

plt.figure(figsize=(7, 6))
plt.scatter(y_test, predictions, alpha=0.35)
limits = [
    min(float(y_test.min()), float(np.min(predictions))),
    max(float(y_test.max()), float(np.max(predictions))),
]
plt.plot(limits, limits, linestyle="--")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title(f"Actual vs Predicted Price — {best_model_name}")
plt.tight_layout()
plt.show()

## 7. Prediction helper

The fitted pipeline includes the location encoder and safely handles previously unseen location labels. An unseen location is encoded as all zeros for the location feature; predictions for such locations may be less reliable.

In [ ]:
def predict_price(size, total_sqft, bath, location, model=best_model):
    input_df = pd.DataFrame([{
        "location": str(location).strip(),
        "size": float(size),
        "total_sqft": float(total_sqft),
        "bath": float(bath),
    }])
    return float(model.predict(input_df)[0])

# Example only; replace these inputs with a property you want to evaluate.
example_prediction = predict_price(3, 1425, 2, "Devarachikkanahalli")
print(f"Example predicted price (source dataset units): {example_prediction:.2f}")

## 8. Save model and evaluation artifacts

The model is saved as a complete pipeline, including preprocessing. Cross-validation results and final held-out test metrics are exported separately.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, OUTPUT_DIR / "price_prediction_pipeline.joblib")
cv_results.to_csv(OUTPUT_DIR / "cross_validation_results.csv", index=False)
test_metrics.to_csv(OUTPUT_DIR / "test_evaluation.csv", index=False)
with open(OUTPUT_DIR / "model_metadata.json", "w", encoding="utf-8") as file:
    json.dump({
        "selected_model": best_model_name,
        "feature_columns": feature_columns,
        "target_column": target_column,
        "target_units": "same units as the source dataset's price column",
        "selection_method": "highest mean 5-fold cross-validation R2 on training data",
    }, file, indent=2)
print(f"Model and evaluation artifacts saved under: {OUTPUT_DIR.resolve()}")